In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/city_day_clean.csv", parse_dates=["Date"])

print(df.shape)
df.dtypes

(29531, 19)


City                        str
Date             datetime64[us]
PM2.5                   float64
PM10                    float64
NO                      float64
NO2                     float64
NOx                     float64
NH3                     float64
CO                      float64
SO2                     float64
O3                      float64
Benzene                 float64
Toluene                 float64
AQI                     float64
AQI_Bucket                  str
Year                      int64
Month                     int64
Season                      str
AQI_Above_500              bool
dtype: object

In [2]:
# Sort so each city's days are in order (needed for lag features)
df = df.sort_values(["City", "Date"]).reset_index(drop=True)

# 1. Calendar features
df["DayOfWeek"] = df["Date"].dt.dayofweek          # 0 = Monday, 6 = Sunday
df["IsWeekend"] = (df["DayOfWeek"] >= 5).astype(int)  # 1 if Saturday/Sunday
df["DayOfYear"] = df["Date"].dt.dayofyear          # 1 to 366

# 2. Yesterday's AQI for the same city (lag feature)
df["AQI_Yesterday"] = df.groupby("City")["AQI"].shift(1)

# 3. Check
print(df[["City", "Date", "AQI", "AQI_Yesterday", "DayOfWeek", "IsWeekend"]].head(6))
print(df["IsWeekend"].value_counts())

        City       Date  AQI  AQI_Yesterday  DayOfWeek  IsWeekend
0  Ahmedabad 2015-01-01  NaN            NaN          3          0
1  Ahmedabad 2015-01-02  NaN            NaN          4          0
2  Ahmedabad 2015-01-03  NaN            NaN          5          1
3  Ahmedabad 2015-01-04  NaN            NaN          6          1
4  Ahmedabad 2015-01-05  NaN            NaN          0          0
5  Ahmedabad 2015-01-06  NaN            NaN          1          0
IsWeekend
0    21098
1     8433
Name: count, dtype: int64


In [3]:
delhi_check = df[df["City"] == "Delhi"][["Date", "AQI", "AQI_Yesterday"]]
print(delhi_check.iloc[100:106])

# How strongly is today's AQI related to yesterday's?
print(df[["AQI", "AQI_Yesterday"]].corr().round(2))

            Date    AQI  AQI_Yesterday
10329 2015-04-11  375.0          290.0
10330 2015-04-12  293.0          375.0
10331 2015-04-13  247.0          293.0
10332 2015-04-14  232.0          247.0
10333 2015-04-15  316.0          232.0
10334 2015-04-16  295.0          316.0
                AQI  AQI_Yesterday
AQI            1.00           0.91
AQI_Yesterday  0.91           1.00


In [4]:
# 1. Rolling 7-day average of PM2.5 for each city (uses only past days)
df["PM25_7day_avg"] = (
    df.groupby("City")["PM2.5"]
    .transform(lambda s: s.shift(1).rolling(window=7, min_periods=3).mean())
)

# 2. Change in AQI from yesterday to today's previous day (a trend signal)
df["AQI_Prev2"] = df.groupby("City")["AQI"].shift(2)
df["AQI_Trend"] = df["AQI_Yesterday"] - df["AQI_Prev2"]

# 3. Check Delhi
cols = ["Date", "PM2.5", "PM25_7day_avg", "AQI_Yesterday", "AQI_Prev2", "AQI_Trend"]
print(df[df["City"] == "Delhi"][cols].iloc[100:106])

            Date   PM2.5  PM25_7day_avg  AQI_Yesterday  AQI_Prev2  AQI_Trend
10329 2015-04-11  131.39      83.204286          290.0      293.0       -3.0
10330 2015-04-12   75.35      93.037143          375.0      290.0       85.0
10331 2015-04-13   86.49      92.571429          293.0      375.0      -82.0
10332 2015-04-14   77.07      87.827143          247.0      293.0      -46.0
10333 2015-04-15   98.79      89.395714          232.0      247.0      -15.0
10334 2015-04-16   81.81      89.888571          316.0      232.0       84.0


## Features created

| Feature | How it is made | Why it may help |
|---|---|---|
| DayOfWeek | day number from Date (0 = Monday) | traffic changes by weekday |
| IsWeekend | 1 if Saturday or Sunday | less traffic and industrial activity on weekends |
| DayOfYear | day number in the year (1 to 366) | captures the seasonal wave (winter peaks, monsoon dips) |
| AQI_Yesterday | AQI shifted down one day, per city | correlation with today's AQI is 0.91 |
| PM25_7day_avg | average PM2.5 of the previous 7 days | smooths out noisy daily readings into a recent level|
| AQI_Trend | yesterday's AQI minus the day before's | shows whether pollution is rising or falling |

In [5]:
df.to_csv("../data/city_day_features.csv", index=False)
print(df.shape)
print(df.columns.tolist())

(29531, 26)
['City', 'Date', 'PM2.5', 'PM10', 'NO', 'NO2', 'NOx', 'NH3', 'CO', 'SO2', 'O3', 'Benzene', 'Toluene', 'AQI', 'AQI_Bucket', 'Year', 'Month', 'Season', 'AQI_Above_500', 'DayOfWeek', 'IsWeekend', 'DayOfYear', 'AQI_Yesterday', 'PM25_7day_avg', 'AQI_Prev2', 'AQI_Trend']


## ML Plan

**Task:** predict a city's AQI for a day using only information from before that day (regression).
**Baseline:** today's AQI = yesterday's AQI.
**Metric:** MAE (mean absolute error), the average size of the mistakes in AQI points.
**Split:** train on 2015-2019, test on 2020, because a random split would leak the future into training. Time-ordered data must be split by time, so the model learns from the past and is tested on later days.
**Not allowed as inputs:** today's PM2.5/PM10 and AQI_Bucket, because they are today's values or are derived from the answer (AQI), which would be data leakage.